# cisTopic Model Analysis

Interactive notebook for exploring a **selected** topic model.

**Prerequisites:** You should have already:
1. Built a cisTopic object (`build_cistopic_obj.py`)
2. Trained models (`run_models.py`)
3. Evaluated models (`evaluate_models.py`) and chosen your topic count

Fill in the config cell below and run all cells.

## 1. Configuration

In [ ]:
# ============================================================
# CONFIGURATION — Fill these in before running
# ============================================================

cistopic_obj_path = ""          # Path to cisTopic .pkl from build_cistopic_obj.py
models_path = ""                # Path to models .pkl from run_models.py (file or directory)
output_dir = ""                 # Output directory for analysis results
selected_n_topics = 0           # Number of topics to use (from evaluate_models.py)
groupby_variable = None         # Cell metadata column for grouping (e.g. "cell_type")
group_colors = None             # Optional dict of {group_name: color} for custom colors

## 2. Imports & Load Objects

In [ ]:
import glob
import os
import pickle

import anndata as ad
import matplotlib.pyplot as plt
import pandas as pd
import scanpy as sc
import seaborn as sns

from pycisTopic.cistopic_class import CistopicObject
from pycisTopic.lda_models import evaluate_models
from pycisTopic.clust_vis import run_umap, plot_metadata, cell_topic_heatmap

# Validate config
assert cistopic_obj_path, "Set cistopic_obj_path in the config cell"
assert models_path, "Set models_path in the config cell"
assert output_dir, "Set output_dir in the config cell"
assert isinstance(selected_n_topics, int) and selected_n_topics > 0, (
    "selected_n_topics must be a positive integer"
)

os.makedirs(output_dir, exist_ok=True)
print(f"Output directory: {output_dir}")

In [ ]:
# Load cisTopic object
print(f"Loading cisTopic object from {cistopic_obj_path}")
with open(cistopic_obj_path, "rb") as f:
    cistopic_obj = pickle.load(f)
print(f"  {cistopic_obj.fragment_matrix.shape[0]} regions x {cistopic_obj.fragment_matrix.shape[1]} cells")
cistopic_obj.cell_data.head()

In [ ]:
# Load models — handles both single .pkl (list) and directory of .pkl files
if os.path.isfile(models_path):
    print(f"Loading models from file: {models_path}")
    with open(models_path, "rb") as f:
        models = pickle.load(f)
    if not isinstance(models, list):
        models = [models]
elif os.path.isdir(models_path):
    print(f"Loading models from directory: {models_path}")
    models = []
    for pkl_file in sorted(glob.glob(os.path.join(models_path, "*.pkl"))):
        m = pickle.load(open(pkl_file, "rb"))
        if isinstance(m, list):
            models.extend(m)
        else:
            models.append(m)
else:
    raise FileNotFoundError(f"Models path not found: {models_path}")

available_topics = sorted([m.cell_topic.shape[0] for m in models])
print(f"  Loaded {len(models)} model(s) with topic counts: {available_topics}")

# Validate selected topic count exists
assert selected_n_topics in available_topics, (
    f"selected_n_topics={selected_n_topics} not found. "
    f"Available: {available_topics}"
)

## 3. Model Evaluation Metrics

In [ ]:
# Plot standard evaluation metrics across all models
# Note: Minmo_2011 (coherence) can produce NaN with very few topic counts or
# sparse data. If it fails, we retry without that metric.
try:
    model = evaluate_models(
        models,
        select_model=None,
        return_model=False,
        metrics=["Arun_2010", "Cao_Juan_2009", "Minmo_2011", "loglikelihood"],
        plot_metrics=True,
        plot=False,
    )
except (ValueError, FloatingPointError):
    print("Minmo_2011 metric failed (likely too few models). Retrying without it.")
    plt.close("all")
    model = evaluate_models(
        models,
        select_model=None,
        return_model=False,
        metrics=["Arun_2010", "Cao_Juan_2009", "loglikelihood"],
        plot_metrics=True,
        plot=False,
    )
plt.savefig(
    os.path.join(output_dir, "model_evaluation_metrics.pdf"),
    bbox_inches="tight", dpi=150,
)
plt.show()

## 4. Select & Add Model to cisTopic Object

In [ ]:
# Select and add the chosen model
model = evaluate_models(
    models,
    select_model=selected_n_topics,
    return_model=True,
    metrics=["Arun_2010", "Cao_Juan_2009", "Minmo_2011", "loglikelihood"],
    plot_metrics=False,
)
cistopic_obj.add_LDA_model(model)
print(f"Selected model with {selected_n_topics} topics")

# Extract cell-topic matrix for the selected model
cell_topic_df = cistopic_obj.selected_model.cell_topic.copy()
cell_topic_df.columns = (
    cell_topic_df.columns.str.split("___", expand=True).get_level_values(0)
)
print(f"Cell-topic matrix: {cell_topic_df.shape} (topics x cells)")

## 5. UMAP Visualization

In [ ]:
# Run cisTopic UMAP on the selected model
run_umap(cistopic_obj, target="cell", scale=False)

# Build AnnData for scanpy-style analysis (cells x topics)
adata_topics = ad.AnnData(
    X=cell_topic_df.values.T,
    obs=pd.DataFrame(index=cell_topic_df.columns),
    var=pd.DataFrame(index=cell_topic_df.index),
)

# Add cisTopic UMAP coordinates
umap_df = cistopic_obj.projections["cell"]["UMAP"].copy()
umap_df.index = umap_df.index.str.split("___", expand=True).get_level_values(0)
adata_topics.obsm["X_umap"] = umap_df.loc[adata_topics.obs_names].values

# Add topic embedding
adata_topics.obsm["X_cisTopic"] = cell_topic_df.values.T

# Copy over cell metadata from cisTopic object
cell_data = cistopic_obj.cell_data.copy()
cell_data.index = cell_data.index.str.split("___", expand=True).get_level_values(0)
for col in cell_data.columns:
    if col in adata_topics.obs_names:
        continue
    matched = cell_data[col].reindex(adata_topics.obs_names)
    if matched.notna().any():
        adata_topics.obs[col] = matched.values

print(adata_topics)
adata_topics.obs.head()

In [ ]:
# Validate groupby variable if set
if groupby_variable is not None:
    assert groupby_variable in adata_topics.obs.columns, (
        f"groupby_variable='{groupby_variable}' not found in cell metadata. "
        f"Available columns: {list(adata_topics.obs.columns)}"
    )
    # Apply custom colors if provided
    if group_colors is not None:
        adata_topics.obs[groupby_variable] = adata_topics.obs[groupby_variable].astype("category")
        present_colors = {k: v for k, v in group_colors.items() if k in adata_topics.obs[groupby_variable].cat.categories}
        adata_topics.obs[groupby_variable] = adata_topics.obs[groupby_variable].cat.reorder_categories(present_colors.keys())
        adata_topics.uns[f"{groupby_variable}_colors"] = list(present_colors.values())

In [ ]:
# Plot UMAP
plot_cols = []
if groupby_variable is not None:
    plot_cols.append(groupby_variable)
plot_cols.extend([c for c in adata_topics.obs.columns if "cisTopic" in c][:2])
if not plot_cols:
    plot_cols = [adata_topics.obs.columns[0]] if len(adata_topics.obs.columns) > 0 else []

if plot_cols:
    fig = sc.pl.umap(
        adata_topics, color=plot_cols, s=40, wspace=0.3, ncols=3,
        show=False, return_fig=True,
    )
    fig.savefig(
        os.path.join(output_dir, "umap_overview.pdf"),
        bbox_inches="tight", dpi=150,
    )
    plt.show()

## 6. Cell-Topic Heatmap

In [ ]:
if groupby_variable is not None:
    ax = sc.pl.heatmap(
        adata_topics,
        var_names=adata_topics.var_names,
        groupby=groupby_variable,
        use_raw=False,
        log=False,
        figsize=(20, 10),
        cmap="viridis",
        show=False,
        show_gene_labels=True,
    )
    plt.savefig(
        os.path.join(output_dir, "cell_topic_heatmap.pdf"),
        bbox_inches="tight", dpi=150,
    )
    plt.show()
else:
    print("Set groupby_variable in the config cell to generate a grouped heatmap.")
    print("Showing ungrouped heatmap instead.")
    fig, ax = plt.subplots(figsize=(20, 10))
    sns.heatmap(adata_topics.to_df().T, cmap="viridis", ax=ax, xticklabels=False)
    ax.set_xlabel("Cells")
    ax.set_ylabel("Topics")
    plt.savefig(
        os.path.join(output_dir, "cell_topic_heatmap.pdf"),
        bbox_inches="tight", dpi=150,
    )
    plt.show()

## 7. Scanpy-Style Analysis

Build a neighbors graph from the topic embedding, run Leiden clustering, and visualize with matrix plots.

In [ ]:
# Add scaled layer for visualization
adata_topics.layers["scaled"] = sc.pp.scale(adata_topics, copy=True).X

# Neighbors + Leiden clustering
sc.pp.neighbors(adata_topics, use_rep="X_cisTopic", n_neighbors=30, metric="cosine")
for res in [0.2, 0.5, 0.8, 1.0]:
    sc.tl.leiden(adata_topics, resolution=res, key_added=f"leiden_{res}")

print("Leiden clustering results added.")

In [ ]:
# Matrix plot — column-scaled topic membership
group_col = groupby_variable if groupby_variable else "leiden_0.5"
sc.pl.matrixplot(
    adata_topics,
    var_names=adata_topics.var_names,
    groupby=group_col,
    dendrogram=True,
    cmap="Blues",
    standard_scale="var",
    colorbar_title="column scaled\ntopic membership",
    figsize=(20, 4),
    save=False,
    show=False,
)
plt.savefig(
    os.path.join(output_dir, "matrixplot_scaled.pdf"),
    bbox_inches="tight", dpi=150,
)
plt.show()

In [ ]:
# Matrix plot — z-score
sc.pl.matrixplot(
    adata_topics,
    var_names=adata_topics.var_names,
    groupby=group_col,
    dendrogram=True,
    colorbar_title="mean z-score",
    layer="scaled",
    vmin=-1,
    vmax=1,
    cmap="RdBu_r",
    save=False,
    show=False,
)
plt.savefig(
    os.path.join(output_dir, "matrixplot_zscore.pdf"),
    bbox_inches="tight", dpi=150,
)
plt.show()

In [ ]:
# Marker topic analysis
sc.tl.rank_genes_groups(adata_topics, groupby=group_col, method="wilcoxon", n_genes=10)
sc.pl.rank_genes_groups(adata_topics, n_genes=10, sharey=False, show=False)
plt.savefig(
    os.path.join(output_dir, "marker_topics.pdf"),
    bbox_inches="tight", dpi=150,
)
plt.show()

## 8. Topic Binarization (Otsu Method)

In [ ]:
from pycisTopic.topic_binarization import binarize_topics

# Binarize topics using Otsu's method
region_bin_topics_otsu = binarize_topics(
    cistopic_obj, method="otsu", plot=True, num_columns=5,
)
plt.savefig(
    os.path.join(output_dir, "topic_binarization_otsu.pdf"),
    bbox_inches="tight", dpi=150,
)
plt.show()

print(f"Binarized {len(region_bin_topics_otsu)} topics")
for topic, regions_df in region_bin_topics_otsu.items():
    print(f"  {topic}: {len(regions_df)} regions")

## 9. Save Final Objects

In [ ]:
# Save cisTopic object with selected model
obj_path = os.path.join(output_dir, "cistopic_obj_with_model.pkl")
print(f"Saving cisTopic object to {obj_path}")
with open(obj_path, "wb") as f:
    pickle.dump(cistopic_obj, f)

# Save topic AnnData — ensure obs dtypes are h5ad-compatible
import numpy as np
adata_save = adata_topics.copy()
for col in adata_save.obs.columns:
    dtype = adata_save.obs[col].dtype
    if hasattr(dtype, "categories"):
        # Already categorical — h5ad handles these natively
        continue
    if dtype == object:
        adata_save.obs[col] = adata_save.obs[col].astype(str)

h5ad_path = os.path.join(output_dir, f"topics_{selected_n_topics}.h5ad")
print(f"Saving topic AnnData to {h5ad_path}")
adata_save.write_h5ad(h5ad_path)

# Save binarized topics
bin_path = os.path.join(output_dir, "binarized_topics_otsu.pkl")
print(f"Saving binarized topics to {bin_path}")
with open(bin_path, "wb") as f:
    pickle.dump(region_bin_topics_otsu, f)

print("\nAll outputs saved to:", output_dir)
for fname in sorted(os.listdir(output_dir)):
    fpath = os.path.join(output_dir, fname)
    if os.path.isfile(fpath):
        size_mb = os.path.getsize(fpath) / 1024 / 1024
        print(f"  {fname} ({size_mb:.1f} MB)")